In [ ]:
from pathlib import Path
from itertools import combinations
import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import spearmanr, pearsonr
from joblib import Parallel, delayed

In [ ]:
# ============================================================
# 1. Paths and settings
# ============================================================

PROJECT_DIR = Path(".")

# Original sampling analysis.
BASELINE_DIR = PROJECT_DIR / "distributions4piton"

# Contains one subfolder for each ACHR start/configuration.
SENSITIVITY_ROOT = (
    PROJECT_DIR / "distributions4piton_ACHR_sensitivity"
)

OUTPUT_DIR = PROJECT_DIR / "icc_ACHR_stability"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Three files can be processed simultaneously.
N_JOBS = 3

# Read a limited number of reactions at a time.
# This prevents loading an entire large sampling CSV into memory.
REACTION_CHUNK_SIZE = 250

# Same cutoff used before original notebook.
HIGH_ICC_THRESHOLD = 0.50

# Set to True if the underlying full CSV files have changed.
RECALCULATE_CACHED_SUMMARIES = False

In [ ]:
# ============================================================
# 2. Discover runs and files
# ============================================================

run_directories = {}

# Add the original analysis as the baseline.
if BASELINE_DIR.exists():
    run_directories["baseline"] = BASELINE_DIR
else:
    print("Baseline directory was not found:", BASELINE_DIR)

# Locate every directory containing sensitivity sampling files.
sensitivity_files = sorted(
    SENSITIVITY_ROOT.rglob("FAC*_full.csv")
)

if not sensitivity_files:
    raise FileNotFoundError(
        f"No FAC*_full.csv files found below:\n"
        f"{SENSITIVITY_ROOT}"
    )

sensitivity_directories = sorted({
    path.parent
    for path in sensitivity_files
})

for directory in sensitivity_directories:

    relative_name = directory.relative_to(
        SENSITIVITY_ROOT
    )

    if str(relative_name) == ".":
        run_name = "sensitivity_root"
    else:
        # A nested path such as configuration_1/start_1 becomes
        # configuration_1__start_1.
        run_name = "__".join(relative_name.parts)

    run_directories[run_name] = directory


# Build run -> mouse -> file mapping.
files_by_run = {}

for run_name, run_directory in run_directories.items():

    files = sorted(
        run_directory.glob("FAC*_full.csv")
    )

    if not files:
        continue

    mouse_files = {
        path.name.replace("_full.csv", ""): path
        for path in files
    }

    files_by_run[run_name] = mouse_files

    print(
        f"{run_name}: "
        f"{len(mouse_files)} mouse sampling files"
    )


if len(files_by_run) < 2:
    raise ValueError(
        "At least two sampling runs are needed."
    )


# Use only mice available in every run.
common_mice = sorted(
    set.intersection(
        *[
            set(mouse_files)
            for mouse_files in files_by_run.values()
        ]
    )
)

if len(common_mice) < 2:
    raise ValueError(
        "Fewer than two mice are shared by all runs."
    )

print(
    f"\nUsing {len(common_mice)} mice present in every run:"
)

print(common_mice)

In [ ]:
# ============================================================
# 3. Summarize one sampling CSV
# ============================================================

CACHE_DIR = OUTPUT_DIR / "per_mouse_summary_cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)


def summarize_sampling_file(
    csv_path,
    cache_path,
    chunk_size=250,
    recalculate=False,
):
    """
    Calculate the mean and within-chain variance of each reaction.

    The CSV is read in reaction chunks so that the complete sampling
    matrix does not need to be loaded into memory.

    Variance uses ddof=0, matching np.nanstd() in the original
    summary-statistics notebook.
    """

    if cache_path.exists() and not recalculate:
        result = pd.read_csv(
            cache_path,
            index_col=0,
        )

        result.index = result.index.astype(str)
        return result

    result_parts = []

    csv_iterator = pd.read_csv(
        csv_path,
        index_col=0,
        chunksize=chunk_size,
    )

    for chunk in csv_iterator:

        chunk.index = (
            chunk.index.astype(str).str.strip()
        )

        values = chunk.to_numpy(dtype=float)

        with np.errstate(
            invalid="ignore",
            divide="ignore",
        ):
            reaction_mean = np.nanmean(
                values,
                axis=1,
            )

            reaction_variance = np.nanvar(
                values,
                axis=1,
                ddof=0,
            )

        result_parts.append(
            pd.DataFrame(
                {
                    "mean": reaction_mean,
                    "variance": reaction_variance,
                },
                index=chunk.index,
            )
        )

    result = pd.concat(result_parts)

    if result.index.duplicated().any():
        duplicated = result.index[
            result.index.duplicated()
        ].unique()

        raise ValueError(
            f"Duplicated reactions in {csv_path.name}: "
            f"{duplicated[:10].tolist()}"
        )

    cache_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    result.to_csv(cache_path)

    return result

In [ ]:
# ============================================================
# Calculate reaction ICC independently for each sampling run
# using only mice present in every run
# ============================================================

icc_by_run = {}
component_tables = {}

for run_name, mouse_files in files_by_run.items():

    print(f"\nCalculating ICC for: {run_name}")

    # Files for the mice shared by every run.
    selected_paths = [
        mouse_files[mouse_id]
        for mouse_id in common_mice
    ]

    cache_paths = [
        (
            CACHE_DIR
            / run_name
            / f"{mouse_id}_sampling_summary.csv"
        )
        for mouse_id in common_mice
    ]

    # This variable is defined here before it is used below.
    mouse_summaries = Parallel(
        n_jobs=N_JOBS,
        backend="loky",
        verbose=5,
    )(
        delayed(summarize_sampling_file)(
            csv_path,
            cache_path,
            REACTION_CHUNK_SIZE,
            RECALCULATE_CACHED_SUMMARIES,
        )
        for csv_path, cache_path
        in zip(selected_paths, cache_paths)
    )

    if len(mouse_summaries) != len(common_mice):
        raise RuntimeError(
            f"{run_name}: expected "
            f"{len(common_mice)} summaries but obtained "
            f"{len(mouse_summaries)}."
        )

    # Store the aligned means and variances for each mouse.
    mouse_means = {}
    mouse_variances = {}

    for mouse_id, summary_df in zip(
        common_mice,
        mouse_summaries,
    ):

        mouse_means[mouse_id] = (
            summary_df["mean"]
        )

        mouse_variances[mouse_id] = (
            summary_df["variance"]
        )

    # Rows = mice; columns = reactions.
    # join="inner" retains reactions available for every mouse
    # within this particular sampling run.
    mean_matrix = pd.concat(
        mouse_means,
        axis=1,
        join="inner",
    ).T

    variance_matrix = pd.concat(
        mouse_variances,
        axis=1,
        join="inner",
    ).T

    # Ensure that both matrices contain the same reactions.
    shared_reactions = (
        mean_matrix.columns.intersection(
            variance_matrix.columns,
            sort=False,
        )
    )

    mean_matrix = mean_matrix.loc[
        :,
        shared_reactions,
    ]

    variance_matrix = variance_matrix.loc[
        :,
        shared_reactions,
    ]

    # Between-mouse variance of the mouse-specific means.
    between_variance = mean_matrix.var(
        axis=0,
        ddof=1,
    )

    # Mean within-mouse sampling variance.
    within_variance = variance_matrix.mean(
        axis=0,
    )

    denominator = (
        between_variance + within_variance
    )

    reaction_icc = (
        between_variance / denominator
    )

    # Leave reactions without any variance as NaN.
    reaction_icc = reaction_icc.where(
        denominator > 0
    ).clip(0, 1)

    reaction_icc.name = run_name

    icc_by_run[run_name] = reaction_icc

    component_table = pd.DataFrame({
        "between_mouse_variance":
            between_variance,
        "mean_within_mouse_variance":
            within_variance,
        "ICC":
            reaction_icc,
    })

    component_table.index.name = "reaction_id"

    component_tables[run_name] = (
        component_table
    )

    component_table.to_csv(
        OUTPUT_DIR
        / f"{run_name}_reaction_ICC_components.csv"
    )

    print(
        f"  Mice used: {mean_matrix.shape[0]}"
    )

    print(
        f"  Reactions considered: "
        f"{mean_matrix.shape[1]}"
    )

    print(
        f"  Defined ICC values: "
        f"{reaction_icc.notna().sum()}"
    )


# Combine the independently calculated ICC vectors.
icc_matrix = pd.concat(
    icc_by_run,
    axis=1,
    join="inner",
)

icc_matrix.index.name = "reaction_id"
icc_matrix.columns.name = "sampling_run"

icc_matrix.to_csv(
    OUTPUT_DIR
    / "reaction_ICC_by_sampling_run.csv"
)

print("\nCombined ICC matrix:", icc_matrix.shape)
display(icc_matrix.head())

In [ ]:
# ============================================================
# 5. Stability of each reaction across all runs
# ============================================================

if "baseline" in icc_matrix.columns:
    baseline_name = "baseline"
else:
    baseline_name = icc_matrix.columns[0]

    print(
        f'No run named "baseline" was found. '
        f'Using "{baseline_name}" as the reference.'
    )


reaction_stability = pd.DataFrame(
    index=icc_matrix.index
)

reaction_stability["n_runs_with_ICC"] = (
    icc_matrix.notna().sum(axis=1)
)

reaction_stability["mean_ICC"] = (
    icc_matrix.mean(axis=1)
)

reaction_stability["median_ICC"] = (
    icc_matrix.median(axis=1)
)

reaction_stability["ICC_standard_deviation"] = (
    icc_matrix.std(axis=1, ddof=1)
)

reaction_stability["minimum_ICC"] = (
    icc_matrix.min(axis=1)
)

reaction_stability["maximum_ICC"] = (
    icc_matrix.max(axis=1)
)

reaction_stability["ICC_range"] = (
    reaction_stability["maximum_ICC"]
    - reaction_stability["minimum_ICC"]
)

reaction_stability["baseline_ICC"] = (
    icc_matrix[baseline_name]
)

# Maximum difference from the selected baseline.
absolute_difference_from_baseline = (
    icc_matrix.subtract(
        icc_matrix[baseline_name],
        axis=0,
    ).abs()
)

reaction_stability[
    "maximum_absolute_difference_from_baseline"
] = absolute_difference_from_baseline.max(axis=1)


# Stability of reaction ranks across runs.
percentile_ranks = icc_matrix.rank(
    axis=0,
    method="average",
    pct=True,
)

reaction_stability[
    "ICC_percentile_rank_standard_deviation"
] = percentile_ranks.std(
    axis=1,
    ddof=1,
)


# Check whether classification as ICC > 0.5 agrees with baseline.
high_icc_status = (
    icc_matrix >= HIGH_ICC_THRESHOLD
)

valid_status = icc_matrix.notna()

baseline_high = high_icc_status[
    baseline_name
]

agreement_with_baseline = (
    high_icc_status.eq(
        baseline_high,
        axis=0,
    )
    & valid_status
)

reaction_stability[
    "fraction_runs_agreeing_with_baseline_class"
] = (
    agreement_with_baseline.sum(axis=1)
    / valid_status.sum(axis=1)
)


reaction_stability = reaction_stability.sort_values(
    [
        "ICC_range",
        "ICC_standard_deviation",
    ],
    ascending=False,
)

reaction_stability.to_csv(
    OUTPUT_DIR
    / "reaction_level_ICC_stability.csv"
)

print(
    "\nMost variable reaction ICC values:"
)

display(
    reaction_stability.head(30)
)

In [ ]:
# ============================================================
# 6. Run-level comparison with baseline
# ============================================================

comparison_records = []

baseline_values = icc_matrix[baseline_name]

for run_name in icc_matrix.columns:

    if run_name == baseline_name:
        continue

    run_values = icc_matrix[run_name]

    valid = (
        baseline_values.notna()
        & run_values.notna()
    )

    baseline_valid = baseline_values[valid]
    run_valid = run_values[valid]

    if valid.sum() >= 3:

        pearson_r = pearsonr(
            baseline_valid,
            run_valid,
        ).statistic

        spearman_rho = spearmanr(
            baseline_valid,
            run_valid,
        ).statistic

        difference = (
            run_valid - baseline_valid
        )

        median_absolute_difference = (
            difference.abs().median()
        )

        mean_absolute_difference = (
            difference.abs().mean()
        )

        rmse = np.sqrt(
            np.mean(difference**2)
        )

    else:
        pearson_r = np.nan
        spearman_rho = np.nan
        median_absolute_difference = np.nan
        mean_absolute_difference = np.nan
        rmse = np.nan

    baseline_high_set = set(
        baseline_valid[
            baseline_valid >= HIGH_ICC_THRESHOLD
        ].index
    )

    run_high_set = set(
        run_valid[
            run_valid >= HIGH_ICC_THRESHOLD
        ].index
    )

    union = baseline_high_set | run_high_set
    intersection = baseline_high_set & run_high_set

    high_icc_jaccard = (
        len(intersection) / len(union)
        if union
        else np.nan
    )

    comparison_records.append({
        "baseline_run": baseline_name,
        "comparison_run": run_name,
        "n_reactions_compared": int(valid.sum()),
        "pearson_ICC": pearson_r,
        "spearman_ICC": spearman_rho,
        "mean_absolute_ICC_difference":
            mean_absolute_difference,
        "median_absolute_ICC_difference":
            median_absolute_difference,
        "ICC_RMSE": rmse,
        "baseline_high_ICC_reactions":
            len(baseline_high_set),
        "comparison_high_ICC_reactions":
            len(run_high_set),
        "high_ICC_overlap":
            len(intersection),
        "high_ICC_Jaccard":
            high_icc_jaccard,
    })


baseline_comparison = pd.DataFrame(
    comparison_records
).sort_values(
    "spearman_ICC",
    ascending=False,
)

baseline_comparison.to_csv(
    OUTPUT_DIR
    / "ICC_sampling_runs_vs_baseline.csv",
    index=False,
)

display(baseline_comparison)

In [ ]:
# ============================================================
# 7. Pairwise comparisons between all runs
# ============================================================

pairwise_spearman = icc_matrix.corr(
    method="spearman",
    min_periods=3,
)

pairwise_pearson = icc_matrix.corr(
    method="pearson",
    min_periods=3,
)

pairwise_median_absolute_difference = pd.DataFrame(
    np.nan,
    index=icc_matrix.columns,
    columns=icc_matrix.columns,
)

for run_a, run_b in combinations(
    icc_matrix.columns,
    2,
):

    valid = (
        icc_matrix[run_a].notna()
        & icc_matrix[run_b].notna()
    )

    median_difference = (
        icc_matrix.loc[valid, run_a]
        .subtract(
            icc_matrix.loc[valid, run_b]
        )
        .abs()
        .median()
    )

    pairwise_median_absolute_difference.loc[
        run_a,
        run_b,
    ] = median_difference

    pairwise_median_absolute_difference.loc[
        run_b,
        run_a,
    ] = median_difference


np.fill_diagonal(
    pairwise_median_absolute_difference.values,
    0.0,
)

pairwise_spearman.to_csv(
    OUTPUT_DIR
    / "pairwise_ICC_spearman_correlations.csv"
)

pairwise_pearson.to_csv(
    OUTPUT_DIR
    / "pairwise_ICC_pearson_correlations.csv"
)

pairwise_median_absolute_difference.to_csv(
    OUTPUT_DIR
    / "pairwise_median_absolute_ICC_difference.csv"
)

In [ ]:
# ============================================================
# 8. ICC distribution comparison
# ============================================================

plt.figure(figsize=(10, 6))

bins = np.linspace(0, 1, 41)

for run_name in icc_matrix.columns:

    values = (
        icc_matrix[run_name]
        .dropna()
        .to_numpy()
    )

    plt.hist(
        values,
        bins=bins,
        histtype="step",
        linewidth=1.8,
        density=True,
        label=run_name,
    )

plt.xlabel("Reaction-level ICC", fontsize=13)
plt.ylabel("Density", fontsize=13)
plt.title("ICC distributions across ACHR sampling runs")
plt.legend(
    fontsize=8,
    frameon=False,
)
plt.tight_layout()

plt.savefig(
    OUTPUT_DIR
    / "ICC_distribution_across_sampling_runs.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()


# Pairwise rank-correlation heatmap.
plt.figure(
    figsize=(
        max(7, 0.8 * len(pairwise_spearman)),
        max(6, 0.7 * len(pairwise_spearman)),
    )
)

sns.heatmap(
    pairwise_spearman,
    vmin=0,
    vmax=1,
    cmap="viridis",
    annot=True,
    fmt=".2f",
    square=True,
)

plt.title(
    "Spearman correlation of reaction ICC values"
)

plt.tight_layout()

plt.savefig(
    OUTPUT_DIR
    / "pairwise_ICC_spearman_heatmap.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()

In [ ]:
# ============================================================
# 9. Baseline-versus-run reaction scatterplots
# ============================================================

alternative_runs = [
    run_name
    for run_name in icc_matrix.columns
    if run_name != baseline_name
]

if alternative_runs:

    n_columns = min(3, len(alternative_runs))
    n_rows = math.ceil(
        len(alternative_runs) / n_columns
    )

    fig, axes = plt.subplots(
        n_rows,
        n_columns,
        figsize=(
            5 * n_columns,
            4.5 * n_rows,
        ),
        squeeze=False,
    )

    for axis, run_name in zip(
        axes.ravel(),
        alternative_runs,
    ):

        valid = (
            icc_matrix[baseline_name].notna()
            & icc_matrix[run_name].notna()
        )

        x = icc_matrix.loc[
            valid,
            baseline_name,
        ]

        y = icc_matrix.loc[
            valid,
            run_name,
        ]

        rho = spearmanr(x, y).statistic

        axis.scatter(
            x,
            y,
            s=8,
            alpha=0.25,
            color="steelblue",
            edgecolors="none",
        )

        axis.plot(
            [0, 1],
            [0, 1],
            linestyle="--",
            color="firebrick",
            linewidth=1,
        )

        axis.set_xlim(0, 1)
        axis.set_ylim(0, 1)

        axis.set_xlabel(
            f"ICC: {baseline_name}"
        )

        axis.set_ylabel(
            f"ICC: {run_name}"
        )

        axis.set_title(
            f"{run_name}\nSpearman ρ = {rho:.3f}"
        )

    # Hide unused axes.
    for axis in axes.ravel()[
        len(alternative_runs):
    ]:
        axis.set_visible(False)

    plt.tight_layout()

    plt.savefig(
        OUTPUT_DIR
        / "reaction_ICC_scatterplots_vs_baseline.png",
        dpi=300,
        bbox_inches="tight",
    )

    plt.show()

# ICC computation across different configurations

In [ ]:
# ============================================================
# 1. Paths and settings
# ============================================================

PROJECT_DIR = Path(".")

# Original analysis containing all 275 mice.
ORIGINAL_BASELINE_DIR = (
    PROJECT_DIR / "distributions4piton_original_20"
)

# Alternative reconstruction configurations containing the 20 mice.
VARIANT_ROOT = (
    PROJECT_DIR
    / "distributions4piton_no_drug_met_variants"
)

OUTPUT_DIR = (
    PROJECT_DIR
    / "icc_ftINIT_configuration_comparison"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CACHE_DIR = OUTPUT_DIR / "per_mouse_summary_cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

N_JOBS = 3
REACTION_CHUNK_SIZE = 250
HIGH_ICC_THRESHOLD = 0.50

# Change to True if the FAC*_full.csv files have changed.
RECALCULATE_CACHED_SUMMARIES = False


# ============================================================
# 2. Define the configurations
# ============================================================

configuration_directories = {
    "original_baseline": ORIGINAL_BASELINE_DIR,

    "ftINIT_1plus0": (
        VARIANT_ROOT / "ftINIT_1plus0"
    ),

    "proteomics_q40_65_90": (
        VARIANT_ROOT / "proteomics_q40_65_90"
    ),

    "transcriptomics_1TPM": (
        VARIANT_ROOT / "transcriptomics_1TPM"
    ),

    "transcriptomics_only_median": (
        VARIANT_ROOT / "transcriptomics_only_median"
    ),
}


# Check that every directory exists.
for configuration, directory in configuration_directories.items():

    if not directory.exists():
        raise FileNotFoundError(
            f"Directory missing for {configuration}:\n"
            f"{directory}"
        )


# ============================================================
# 3. Find mouse files in every configuration
# ============================================================

files_by_configuration = {}

for configuration, directory in configuration_directories.items():

    files = sorted(directory.glob("FAC*_full.csv"))

    if not files:
        raise FileNotFoundError(
            f"No FAC*_full.csv files found for "
            f"{configuration} in:\n{directory}"
        )

    mouse_files = {
        file.name.replace("_full.csv", ""): file
        for file in files
    }

    files_by_configuration[configuration] = mouse_files

    print(
        f"{configuration}: "
        f"{len(mouse_files)} mouse files"
    )


# Use the same mice in every ICC calculation.
common_mice = sorted(
    set.intersection(
        *[
            set(mouse_files)
            for mouse_files
            in files_by_configuration.values()
        ]
    )
)

if len(common_mice) < 2:
    raise ValueError(
        "Fewer than two mice are shared by all configurations."
    )

print(
    f"\nUsing {len(common_mice)} mice shared by all configurations:"
)

print(common_mice)

In [ ]:
# ============================================================
# 4. Summarize one mouse sampling distribution
# ============================================================

def summarize_sampling_file(
    csv_path,
    cache_path,
    chunk_size=250,
    recalculate=False,
):
    """
    Calculate the sampled mean and within-mouse sampling variance
    for every reaction.

    Rows in the input CSV are reactions and columns are sampled
    flux vectors.
    """

    if cache_path.exists() and not recalculate:

        result = pd.read_csv(
            cache_path,
            index_col=0,
        )

        result.index = result.index.astype(str)

        return result

    result_parts = []

    chunks = pd.read_csv(
        csv_path,
        index_col=0,
        chunksize=chunk_size,
    )

    for chunk in chunks:

        chunk.index = (
            chunk.index
            .astype(str)
            .str.strip()
        )

        values = chunk.to_numpy(dtype=float)

        with np.errstate(
            invalid="ignore",
            divide="ignore",
        ):

            reaction_means = np.nanmean(
                values,
                axis=1,
            )

            reaction_variances = np.nanvar(
                values,
                axis=1,
                ddof=0,
            )

        result_parts.append(
            pd.DataFrame(
                {
                    "mean": reaction_means,
                    "variance": reaction_variances,
                },
                index=chunk.index,
            )
        )

    result = pd.concat(result_parts)

    if result.index.duplicated().any():

        duplicated_reactions = result.index[
            result.index.duplicated()
        ].unique()

        raise ValueError(
            f"Duplicated reaction IDs in {csv_path.name}: "
            f"{duplicated_reactions[:10].tolist()}"
        )

    cache_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    result.to_csv(cache_path)

    return result

In [ ]:
# ============================================================
# 5. Calculate ICC independently for each configuration
# ============================================================

icc_by_configuration = {}
icc_component_tables = {}

for configuration, mouse_files in files_by_configuration.items():

    print(
        f"\nCalculating ICC for: {configuration}"
    )

    selected_paths = [
        mouse_files[mouse_id]
        for mouse_id in common_mice
    ]

    cache_paths = [
        (
            CACHE_DIR
            / configuration
            / f"{mouse_id}_summary.csv"
        )
        for mouse_id in common_mice
    ]

    mouse_summaries = Parallel(
        n_jobs=N_JOBS,
        backend="loky",
        verbose=5,
    )(
        delayed(summarize_sampling_file)(
            csv_path,
            cache_path,
            REACTION_CHUNK_SIZE,
            RECALCULATE_CACHED_SUMMARIES,
        )
        for csv_path, cache_path
        in zip(selected_paths, cache_paths)
    )

    # Rows = mice; columns = reactions.
    mean_matrix = pd.concat(
        [
            summary["mean"].rename(mouse_id)
            for mouse_id, summary
            in zip(common_mice, mouse_summaries)
        ],
        axis=1,
        join="inner",
    ).T

    variance_matrix = pd.concat(
        [
            summary["variance"].rename(mouse_id)
            for mouse_id, summary
            in zip(common_mice, mouse_summaries)
        ],
        axis=1,
        join="inner",
    ).T

    shared_reactions = (
        mean_matrix.columns.intersection(
            variance_matrix.columns,
            sort=False,
        )
    )

    mean_matrix = mean_matrix.loc[
        :,
        shared_reactions,
    ]

    variance_matrix = variance_matrix.loc[
        :,
        shared_reactions,
    ]

    # Variance of mouse-specific sampled means.
    between_mouse_variance = mean_matrix.var(
        axis=0,
        ddof=1,
    )

    # Average sampling variance within mice.
    mean_within_mouse_variance = variance_matrix.mean(
        axis=0,
    )

    denominator = (
        between_mouse_variance
        + mean_within_mouse_variance
    )

    reaction_icc = (
        between_mouse_variance
        / denominator
    )

    # Reactions with no between- or within-mouse variance
    # do not have a defined ICC.
    reaction_icc = reaction_icc.where(
        denominator > 0
    ).clip(0, 1)

    reaction_icc.name = configuration

    icc_by_configuration[configuration] = (
        reaction_icc
    )

    component_table = pd.DataFrame(
        {
            "between_mouse_variance":
                between_mouse_variance,

            "mean_within_mouse_variance":
                mean_within_mouse_variance,

            "ICC":
                reaction_icc,
        }
    )

    component_table.index.name = "reaction_id"

    icc_component_tables[configuration] = (
        component_table
    )

    component_table.to_csv(
        OUTPUT_DIR
        / f"{configuration}_ICC_components.csv"
    )

    print(
        f"  Mice: {mean_matrix.shape[0]}"
    )

    print(
        f"  Reactions considered: "
        f"{mean_matrix.shape[1]}"
    )

    print(
        f"  Defined ICC values: "
        f"{reaction_icc.notna().sum()}"
    )


# Combine ICC vectors.
icc_matrix = pd.concat(
    icc_by_configuration,
    axis=1,
    join="outer",
)

icc_matrix.index.name = "reaction_id"
icc_matrix.columns.name = "configuration"

icc_matrix.to_csv(
    OUTPUT_DIR
    / "reaction_ICC_by_configuration.csv"
)

print("\nICC matrix shape:", icc_matrix.shape)

display(icc_matrix.head())

In [ ]:
# ============================================================
# 6. Summarize each ICC distribution
# ============================================================

distribution_records = []

for configuration in icc_matrix.columns:

    values = (
        icc_matrix[configuration]
        .dropna()
    )

    distribution_records.append(
        {
            "configuration": configuration,
            "number_defined_ICCs": len(values),
            "mean_ICC": values.mean(),
            "median_ICC": values.median(),
            "q25_ICC": values.quantile(0.25),
            "q75_ICC": values.quantile(0.75),
            "minimum_ICC": values.min(),
            "maximum_ICC": values.max(),
            "fraction_ICC_at_least_0_5":
                (values >= HIGH_ICC_THRESHOLD).mean(),
        }
    )

icc_distribution_summary = pd.DataFrame(
    distribution_records
)

icc_distribution_summary.to_csv(
    OUTPUT_DIR
    / "ICC_distribution_summary.csv",
    index=False,
)

display(icc_distribution_summary)

In [ ]:
# ============================================================
# 7. Compare every alternative with the original baseline
# ============================================================

baseline_name = "original_baseline"
baseline_icc = icc_matrix[baseline_name]

comparison_records = []

for configuration in icc_matrix.columns:

    if configuration == baseline_name:
        continue

    alternative_icc = icc_matrix[configuration]

    # Compare only reactions with defined ICC in both configurations.
    valid = (
        baseline_icc.notna()
        & alternative_icc.notna()
    )

    baseline_values = baseline_icc[valid]
    alternative_values = alternative_icc[valid]

    if valid.sum() >= 3:

        pearson_r = pearsonr(
            baseline_values,
            alternative_values,
        ).statistic

        spearman_rho = spearmanr(
            baseline_values,
            alternative_values,
        ).statistic

        differences = (
            alternative_values
            - baseline_values
        )

        mean_absolute_difference = (
            differences.abs().mean()
        )

        median_absolute_difference = (
            differences.abs().median()
        )

        rmse = np.sqrt(
            np.mean(differences**2)
        )

    else:

        pearson_r = np.nan
        spearman_rho = np.nan
        mean_absolute_difference = np.nan
        median_absolute_difference = np.nan
        rmse = np.nan

    # Jaccard similarity of reactions classified as high ICC.
    baseline_high = set(
        baseline_values[
            baseline_values >= HIGH_ICC_THRESHOLD
        ].index
    )

    alternative_high = set(
        alternative_values[
            alternative_values >= HIGH_ICC_THRESHOLD
        ].index
    )

    high_icc_union = (
        baseline_high | alternative_high
    )

    high_icc_intersection = (
        baseline_high & alternative_high
    )

    high_icc_jaccard_similarity = (
        len(high_icc_intersection)
        / len(high_icc_union)
        if high_icc_union
        else np.nan
    )

    high_icc_jaccard_distance = (
        1 - high_icc_jaccard_similarity
        if np.isfinite(
            high_icc_jaccard_similarity
        )
        else np.nan
    )

    comparison_records.append(
        {
            "baseline_configuration":
                baseline_name,

            "alternative_configuration":
                configuration,

            "number_reactions_compared":
                int(valid.sum()),

            "pearson_ICC":
                pearson_r,

            "spearman_ICC":
                spearman_rho,

            "mean_absolute_ICC_difference":
                mean_absolute_difference,

            "median_absolute_ICC_difference":
                median_absolute_difference,

            "ICC_RMSE":
                rmse,

            "baseline_high_ICC_reactions":
                len(baseline_high),

            "alternative_high_ICC_reactions":
                len(alternative_high),

            "high_ICC_overlap":
                len(high_icc_intersection),

            "high_ICC_Jaccard_similarity":
                high_icc_jaccard_similarity,

            "high_ICC_Jaccard_distance":
                high_icc_jaccard_distance,
        }
    )

icc_baseline_comparison = pd.DataFrame(
    comparison_records
).sort_values(
    "spearman_ICC",
    ascending=False,
)

icc_baseline_comparison.to_csv(
    OUTPUT_DIR
    / "ICC_configurations_vs_original_baseline.csv",
    index=False,
)

display(icc_baseline_comparison)

In [ ]:
# ============================================================
# 8. Pairwise ICC comparison between all configurations
# ============================================================

pairwise_spearman = icc_matrix.corr(
    method="spearman",
    min_periods=3,
)

pairwise_pearson = icc_matrix.corr(
    method="pearson",
    min_periods=3,
)

pairwise_median_absolute_difference = pd.DataFrame(
    np.nan,
    index=icc_matrix.columns,
    columns=icc_matrix.columns,
)

for configuration_a in icc_matrix.columns:

    for configuration_b in icc_matrix.columns:

        valid = (
            icc_matrix[configuration_a].notna()
            & icc_matrix[configuration_b].notna()
        )

        difference = (
            icc_matrix.loc[
                valid,
                configuration_a,
            ]
            - icc_matrix.loc[
                valid,
                configuration_b,
            ]
        ).abs()

        pairwise_median_absolute_difference.loc[
            configuration_a,
            configuration_b,
        ] = difference.median()

pairwise_spearman.to_csv(
    OUTPUT_DIR
    / "pairwise_ICC_spearman.csv"
)

pairwise_pearson.to_csv(
    OUTPUT_DIR
    / "pairwise_ICC_pearson.csv"
)

pairwise_median_absolute_difference.to_csv(
    OUTPUT_DIR
    / "pairwise_median_absolute_ICC_difference.csv"
)

display(pairwise_spearman)
display(pairwise_median_absolute_difference)

In [ ]:
# ============================================================
# 9. Reaction-level ICC stability across configurations
# ============================================================

reaction_stability = pd.DataFrame(
    index=icc_matrix.index
)

reaction_stability["number_configurations"] = (
    icc_matrix.notna().sum(axis=1)
)

reaction_stability["mean_ICC"] = (
    icc_matrix.mean(axis=1)
)

reaction_stability["median_ICC"] = (
    icc_matrix.median(axis=1)
)

reaction_stability["ICC_standard_deviation"] = (
    icc_matrix.std(axis=1, ddof=1)
)

reaction_stability["minimum_ICC"] = (
    icc_matrix.min(axis=1)
)

reaction_stability["maximum_ICC"] = (
    icc_matrix.max(axis=1)
)

reaction_stability["ICC_range"] = (
    reaction_stability["maximum_ICC"]
    - reaction_stability["minimum_ICC"]
)

reaction_stability["original_baseline_ICC"] = (
    icc_matrix[baseline_name]
)

reaction_stability[
    "maximum_absolute_difference_from_baseline"
] = (
    icc_matrix
    .subtract(
        icc_matrix[baseline_name],
        axis=0,
    )
    .abs()
    .max(axis=1)
)

reaction_stability = reaction_stability.sort_values(
    [
        "ICC_range",
        "ICC_standard_deviation",
    ],
    ascending=False,
)

reaction_stability.to_csv(
    OUTPUT_DIR
    / "reaction_level_ICC_stability_across_configurations.csv"
)

display(reaction_stability.head(30))

# Visualization

In [ ]:
# ============================================================
# 10. Plot ICC distributions
# ============================================================

display_names = {
    "original_baseline":
        "Original baseline",

    "ftINIT_1plus0":
        "Alternative ftINIT setting",

    "proteomics_q40_65_90":
        "Alternative proteomics thresholds",

    "transcriptomics_1TPM":
        "Transcript threshold: 1 TPM",

    "transcriptomics_only_median":
        "Transcriptomics only",
}

plt.figure(figsize=(10, 6))

bins = np.linspace(0, 1, 41)

for configuration in icc_matrix.columns:

    values = (
        icc_matrix[configuration]
        .dropna()
        .to_numpy()
    )

    plt.hist(
        values,
        bins=bins,
        histtype="step",
        linewidth=1.8,
        density=True,
        label=display_names.get(
            configuration,
            configuration,
        ),
    )

plt.xlabel("Reaction-level ICC", fontsize=12)
plt.ylabel("Density", fontsize=12)

plt.title(
    "ICC distributions across reconstruction configurations"
)

plt.legend(
    frameon=False,
    fontsize=9,
)

plt.tight_layout()

plt.savefig(
    OUTPUT_DIR
    / "ICC_distributions_across_configurations.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()